# Notebook 04 — Task 1.5: Unified Attribution Summaries + **Rules vs Model** Comparison

🎯 **Objective**  
Create a **single pipeline** that pulls together:
- **Rule-based attribution** (First/Last/Linear/Time-Decay),
- **Model-based attribution** (from SHAP, first/last OHE credits),
- **Shapley attribution** (Monte-Carlo over touchpoints, optional), and
- **Spend** (from journeys),
To produce **campaign- and channel-level ROI tables**, a **channel ROI heatmap**, and a **Rules vs Model** share comparison that highlights where last-touch over/under-credits campaigns vs the model.

---

## 🔌 Inputs

- **Rule-based credits**: `../artifacts/bi_exports/campaign_attribution_rules.csv`  
  *(columns: `rule, campaign, credit` — campaign is renamed to `campaign_id`)*
- **Model-based (SHAP) aggregation**: `../artifacts/bi_exports/model_based_attribution_shap.csv`  
  *(columns: `campaign_id, credit_pos_total, credit_raw_total, share_pos, share_raw`)*
- **Shapley campaign credits** *(optional)*: `../artifacts/bi_exports/campaign_attribution_shapley.csv`  
  *(columns: `campaign_id, shapley_credit, share` → share renamed to `shapley_share`)*
- **Journeys (for spend)**: `../data/processed/journeys_all.parquet`  
  *(uses `campaign_seq` + `cost_seq` to sum spend per campaign)*
- **Campaign→Channel map** *(optional, build/editable)*: `../data/meta/campaign_channels.csv`

---

## 🛠️ What the notebook does

### 1) Load & harmonize attribution sources
- **Rules → wide**: pivot to one row per campaign with a column per rule (`first_touch`, `last_touch`, `linear`, `time_decay_d0.5`, …).
- **Model view**: keep both **positive-only** and **raw signed** totals  
  - `model_credit_pos` (sum of positive SHAP at first/last)  
  - `model_credit_raw` (sum of signed SHAP at first/last)
- **Shapley view (optional)**: `shapley_credit`, `shapley_share` if file exists.

### 2) Build **spend per campaign** from journeys
- Parse `campaign_seq` & `cost_seq` (list-safe), **align lengths**, sum costs per campaign → `spend`.

### 3) Merge to a **campaign-level master** + compute ROI
- Union of all campaign IDs across rules/model/shapley/spend.
- Join **channel mapping** if available (else default `(unassigned)`).
- Add ROI **per method**:  
  `roi__X = attributed_conversions_X / spend`  
  *(safe division; 0 if spend is 0)*

### 4) Channel rollups + ROI heatmap
- Aggregate campaigns → **channel-level totals** for every attribution method.
- Compute **channel-level ROI** for each method.
- Export a **wide heatmap** (channel × `roi__*`) for quick BI.

### 5) **Rules vs Model** shares & deltas
- Normalize **rule columns** to shares per rule (sum to 1.0).
- Compare each **rule share** to **model `share_pos`**:
  - `delta_<rule>_vs_shap_pos = share_pos - rule_share`
- Export **top gainers/losers vs last-touch** (who the model thinks deserves more/less share than last-touch).

### 6) Channel mapping helper (build/edit cycle)
- Creates a **draft** `../data/meta/campaign_channels.csv` with:
  - `campaign_id, impressions, clicks, spend, conv_last_touch, channel`
- You fill `channel` labels (e.g., *Search*, *Social*, *Retargeting*), then re-run the **rebuild** cell to refresh all **channel** summaries and the heatmap.

---

## 📂 Outputs

| File | What it contains | Use it for |
|---|---|---|
| `../artifacts/bi_exports/campaign_summary_all.csv` | Campaign-level table with **rule columns**, `model_credit_pos`, `model_credit_raw`, *(optional)* `shapley_credit`, `spend`, `channel`, and **ROI columns** (`roi__*`). | BI tables, pivoting by campaign or channel. |
| `../artifacts/bi_exports/channel_summary_all.csv` | Channel-level **sums** of all credit methods + **ROI by channel**. | Media-mix review by channel. |
| `../artifacts/bi_exports/channel_roi_heatmap.csv` | Channel × `roi__*` **matrix**. | Heatmap viz in BI. |
| `../artifacts/bi_exports/compare_all_rules_vs_shap.csv` | **Rule shares** vs **model share_pos** and **deltas** (e.g., `delta_last_touch_vs_shap_pos`). | Identify campaigns **over/under-credited** by last-touch vs model. |
| `../artifacts/bi_exports/campaign_summary_campaign_only.csv` | Campaign-only version (no channels), with ROI fields. | Lightweight campaign table when channel map is not ready. |
| `../data/meta/campaign_channels.csv` | **Editable** mapping with helper stats. | Tagging campaigns to channels; re-run to update rollups. |

---

## 🔍 Interpreting the columns

- **Rule columns** (e.g., `first_touch`, `last_touch`, `linear`, `time_decay_d0.5`):  
  *Attributed conversion counts* under each heuristic.
- **Model-based**:
  - `model_credit_pos`: positive SHAP (first/last) totals → conservative lift view.
  - `model_credit_raw`: signed SHAP (first/last) totals → allows negatives (penalty).
  - `share_pos`, `share_raw`: model shares (from the model-based CSV).
- **Shapley** *(if present)*:
  - `shapley_credit`, `shapley_share`: Monte-Carlo touchpoint Shapley aggregates.
- **Spend**: sum of impression-level costs attributed to each campaign (from journeys).
- **ROI columns**: `roi__X = X / spend`  
  *(Proxy ROI if you don’t have revenue values; **higher is better**)*
- **Delta columns** (in compare file):  
  `delta_last_touch_vs_shap_pos = share_pos - last_touch_share`  
  - **Positive** → Model assigns **more** share than last-touch (likely under-credited by last-touch).  
  - **Negative** → Model assigns **less** share than last-touch (likely over-credited by last-touch).

---

## 🧭 How to use this in your portfolio/BI

1. **Export & Inspect**  
   - Open `campaign_summary_all.csv` for full campaign view with **ROI by method**.
   - Open `channel_summary_all.csv` + `channel_roi_heatmap.csv` for **channel-level** ROI.
2. **Tag Channels**  
   - Fill `channel` in `../data/meta/campaign_channels.csv` (use helper stats).
   - Re-run the *Rebuild summaries* cell to refresh channel metrics.
3. **Find Misattributions**  
   - Use `compare_all_rules_vs_shap.csv` to spot **gainers/losers vs last-touch**.
   - Campaigns with large **positive delta** are likely **assist/sequence** plays that last-touch misses.
4. **Make Decisions**  
   - Combine ROI + deltas to propose **Scale / Hold / Trim** by campaign/channel in your deck.

---

## ✅ Data hygiene & safeguards

- **List-safe parsing** for sequence columns (works with strings/arrays/nulls).
- **Length alignment** of `campaign_seq` ↔ `cost_seq` to avoid mis-sums.
- **Type harmonization** `campaign_id` → string across all inputs.
- **Zero-safe division** for ROI (`safe_div` → 0 when spend=0).
- **Share normalization**: each rule’s shares sum to 1.0 before comparison.
- Works **with or without** Shapley & channel map (graceful fallbacks).

---

## ⚠️ Notes & good-to-knows

- **ROI here = conversions/spend** (proxy). If you have revenue, replace with value-based ROAS.  
- **Model shares** (`share_pos`) come from your **model-based SHAP aggregation** (Notebook 04); they are **not** recomputed here.
- The second section writes `compare_all_rules_vs_shap.csv` again (campaign-only—but same core columns). That’s expected.

---

## 🔌 How this fits the pipeline
1. **Notebook 01 — Data Preprocessing & EDA**
   Prepare impression-level data, KPIs, and Parquet artifacts.
2. **Notebook 02 — Build User Journeys**  
   Convert impressions → ordered user journeys with sequences.
3. **Notebook 03 — Attribution & Modeling (3a–3c)**  
   - **03a — Heuristic Attribution:** First/Last/Linear/Time-decay over journeys.  
   - **03b — Journey Conversion Modeling:** Logistic Regression & XGBoost with calibration + SHAP.  
   - **03c — Shapley Attribution (Monte Carlo):** Model-based credit per campaign.
4. **Notebook 04 — Attribution Rollups, ROI & Rule-vs-Model Comparison** (This)  
   Consolidate attributions, compute ROI, compare methods.
5. **Notebook 05 — Customer Segmentation & Audience Exports**   
   KMeans segmentation, campaign-mix profiles, and BI exports.
6. **Notebook 06 — Budget Reallocation Simulator (Upcoming)**  
   What-if scenarios using attributed ROI and constraints.
   
---

## 📝 Quick recap (business speak)

This notebook **joins every attribution lens** (rules, model, Shapley) with **spend** to show **who’s truly efficient** (ROI) and where **last-touch disagrees** with the model. It gives you ready-made CSVs for **campaign** and **channel** views, a **heatmap** for slides, and a **delta table** to argue for **budget reallocation** with evidence.


In [1]:
# Rule-vs-Model Comparison (single pipeline, robust spend)
import os, numpy as np, pandas as pd
from IPython.display import display


BI_DIR = "../artifacts/bi_exports"
os.makedirs(BI_DIR, exist_ok=True)

PATH_RULES    = f"{BI_DIR}/campaign_attribution_rules.csv"          # from rule-based notebook
PATH_MODEL    = f"{BI_DIR}/model_based_attribution_shap.csv"        # from model-based (SHAP) notebook
PATH_SHAPLEY  = f"{BI_DIR}/campaign_attribution_shapley.csv"        # from Shapley notebook (
PATH_JOURNEYS = "../data/processed/journeys_all.parquet"            # all journeys (0/1)
PATH_CHMAP    = "../data/meta/campaign_channels.csv"                # campaign_id,channel

# Load attribution outputs 
rules = pd.read_csv(PATH_RULES).rename(columns={"campaign":"campaign_id"})  # [rule, campaign_id, credit]
model = pd.read_csv(PATH_MODEL)                                             # [campaign_id, credit_pos_total, credit_raw_total, share_pos, share_raw]
for df in (rules, model):
    df["campaign_id"] = df["campaign_id"].astype(str)

# Handle Shapley optionality
if os.path.exists(PATH_SHAPLEY):
    shpy = pd.read_csv(PATH_SHAPLEY)
    shpy["campaign_id"] = shpy["campaign_id"].astype(str)
    shpy_view = shpy[["campaign_id","shapley_credit","share"]].rename(columns={"share":"shapley_share"})
else:
    shpy_view = pd.DataFrame(columns=["campaign_id","shapley_credit","shapley_share"])

# Rules → wide (each rule column = attributed conversions under that rule)
rules_wide = (
    rules.groupby(["rule","campaign_id"], as_index=False)["credit"].sum()
         .pivot(index="campaign_id", columns="rule", values="credit")
         .fillna(0.0)
         .reset_index()
)
rules_wide.columns.name = None

# Model view (both positive & raw)
model_view = (
    model[["campaign_id","credit_pos_total","credit_raw_total","share_pos","share_raw"]]
    .rename(columns={"credit_pos_total":"model_credit_pos",
                     "credit_raw_total":"model_credit_raw"})
)

#  Spend per campaign from journeys (ROBUST) 
J = pd.read_parquet(PATH_JOURNEYS)

def to_pylist(x):
    if isinstance(x, list): return x
    if isinstance(x, (np.ndarray, pd.Series)): return x.tolist()
    if isinstance(x, str) and x.strip().startswith('[') and x.strip().endswith(']'):
        import ast
        try: return ast.literal_eval(x)
        except Exception: return []
    if x is None or (isinstance(x, float) and pd.isna(x)) or x == "": return []
    return [x]

for c in ["campaign_seq","cost_seq"]:
    if c in J.columns:
        J[c] = J[c].apply(to_pylist)

def align(a, b, fill=0.0):
    if len(b) < len(a): b = b + [fill]*(len(a)-len(b))
    elif len(b) > len(a): b = b[:len(a)]
    return a, b

has_cost_seq   = "cost_seq" in J.columns
has_total_cost = "total_cost" in J.columns

rows = []
for r in J.itertuples(index=False):
    cs = getattr(r, "campaign_seq", []) or []
    # Prefer per-touch costs from cost_seq when available & aligned
    if has_cost_seq:
        co = getattr(r, "cost_seq", []) or []
        if len(co) < len(cs):
            # Fallback: split total_cost equally across touches
            tot = float(getattr(r, "total_cost", 0.0)) if has_total_cost else 0.0
            per = (tot / len(cs)) if len(cs) else 0.0
            co = [per] * len(cs)
        else:
            cs, co = align(cs, co, 0.0)
    else:
        # No cost_seq at all → fallback to total_cost split
        tot = float(getattr(r, "total_cost", 0.0)) if has_total_cost else 0.0
        per = (tot / len(cs)) if len(cs) else 0.0
        co = [per] * len(cs)

    for c_id, cst in zip(cs, co):
        rows.append((str(c_id), float(cst)))

spend = (
    pd.DataFrame(rows, columns=["campaign_id","cost"])
      .groupby("campaign_id", as_index=False)["cost"].sum()
      .rename(columns={"cost":"spend"})
)

# channel mapping 
if os.path.exists(PATH_CHMAP):
    chmap = pd.read_csv(PATH_CHMAP, dtype={"campaign_id":str})
    if "channel" not in chmap.columns:
        chmap["channel"] = "(unassigned)"
else:
    all_ids_for_map = pd.Index(
        pd.concat([
            rules_wide["campaign_id"],
            model_view["campaign_id"],
            shpy_view.get("campaign_id", pd.Series([], dtype=str)),
            spend["campaign_id"]
        ]).astype(str).unique()
    )
    chmap = pd.DataFrame({"campaign_id": all_ids_for_map, "channel": "(unassigned)"})

# Campaign-level master + ROI 
# union of campaign ids
all_ids = pd.Series(pd.concat([
    rules_wide["campaign_id"],
    model_view["campaign_id"],
    shpy_view.get("campaign_id", pd.Series([], dtype=str)),
    spend["campaign_id"],
]).unique(), name="campaign_id").to_frame()

camp = (all_ids
        .merge(rules_wide, on="campaign_id", how="left")
        .merge(model_view, on="campaign_id", how="left")
        .merge(shpy_view,  on="campaign_id", how="left")
        .merge(spend,      on="campaign_id", how="left")
        .merge(chmap,      on="campaign_id", how="left"))

# Hard guard: ensure 'spend' exists
if "spend" not in camp.columns:
    camp["spend"] = 0.0

# numeric fill
num_cols = camp.columns.difference(["campaign_id","channel"])
camp[num_cols] = camp[num_cols].fillna(0.0)

# attribution “conversion” columns
rule_cols  = [c for c in camp.columns if c not in [
    "campaign_id","channel","spend",
    "model_credit_pos","model_credit_raw",
    "shapley_credit","share_pos","share_raw",
    "shapley_share"
]]
model_cols = [c for c in ["model_credit_pos","model_credit_raw"] if c in camp.columns]
shpy_col   = "shapley_credit" if "shapley_credit" in camp.columns else None

def safe_div(a, b):
    b = b.replace(0, np.nan)
    return (a / b).fillna(0.0)

# ROI = conversions / spend 
for rc in rule_cols:
    camp[f"roi__{rc}"] = safe_div(camp[rc], camp["spend"])

for mc in model_cols:
    camp[f"roi__{mc}"] = safe_div(camp[mc], camp["spend"])

if shpy_col:
    camp["roi__shapley"] = safe_div(camp[shpy_col], camp["spend"])

camp_out_path = f"{BI_DIR}/campaign_summary_all.csv"
camp.to_csv(camp_out_path, index=False)
print("Saved ->", camp_out_path)

#Channel-level rollups + ROI heatmap 
agg_cols_all = rule_cols + model_cols + ([shpy_col] if shpy_col else []) + ["spend"]
agg_cols = [c for c in agg_cols_all if c in camp.columns]  # filter to existing cols

chan = camp.groupby("channel", as_index=False)[agg_cols].sum()

# ROI by channel
for rc in rule_cols:
    if rc in chan.columns:
        chan[f"roi__{rc}"] = safe_div(chan[rc], chan["spend"])
for mc in model_cols:
    if mc in chan.columns:
        chan[f"roi__{mc}"] = safe_div(chan[mc], chan["spend"])
if shpy_col and shpy_col in chan.columns:
    chan["roi__shapley"] = safe_div(chan[shpy_col], chan["spend"])

chan_out_path = f"{BI_DIR}/channel_summary_all.csv"
chan.to_csv(chan_out_path, index=False)
print("Saved ->", chan_out_path)

roi_cols = [c for c in chan.columns if c.startswith("roi__")]
heatmap = chan[["channel"] + roi_cols].set_index("channel")
heatmap_out_path = f"{BI_DIR}/channel_roi_heatmap.csv"
heatmap.to_csv(heatmap_out_path)
print("Saved ->", heatmap_out_path)

# Rules vs SHAP shares + deltas 
# Build rule shares (normalize each rule to 1.0)
rules_shares = rules.copy()
rules_shares["rule_share"] = rules_shares.groupby("rule")["credit"].transform(lambda x: x / x.sum())

# campaigns × rules (wide)
rule_share_wide = (
    rules_shares.pivot(index="campaign_id", columns="rule", values="rule_share")
                .fillna(0.0)
                .reset_index()
)

# join SHAP shares 
cmp = (
    rule_share_wide.merge(model_view[["campaign_id","share_pos","share_raw"]],
                          on="campaign_id", how="left")
                   .fillna(0.0)
)

# build deltas vs SHAP positive share
for col in rule_share_wide.columns.tolist()[1:]:
    cmp[f"delta_{col}_vs_shap_pos"] = cmp["share_pos"] - cmp[col]

# Top gainers/losers vs last_touch 
if "last_touch" in cmp.columns:
    top_gainers = cmp.sort_values("delta_last_touch_vs_shap_pos", ascending=False).head(10)[
        ["campaign_id","share_pos","last_touch","delta_last_touch_vs_shap_pos"]
    ]
    top_losers = cmp.sort_values("delta_last_touch_vs_shap_pos", ascending=True).head(10)[
        ["campaign_id","share_pos","last_touch","delta_last_touch_vs_shap_pos"]
    ]
else:
    top_gainers = pd.DataFrame()
    top_losers  = pd.DataFrame()

cmp_out_path = f"{BI_DIR}/compare_all_rules_vs_shap.csv"
cmp.to_csv(cmp_out_path, index=False)
print("Saved ->", cmp_out_path)

print("\n=== PREVIEW ===")
display(camp.head(5))
display(chan.head(5))
display(heatmap)
if len(top_gainers):
    print("\nTop gainers vs last-touch:")
    display(top_gainers)
if len(top_losers):
    print("\nTop losers vs last-touch:")
    display(top_losers)


Saved -> ../artifacts/bi_exports/campaign_summary_all.csv
Saved -> ../artifacts/bi_exports/channel_summary_all.csv
Saved -> ../artifacts/bi_exports/channel_roi_heatmap.csv
Saved -> ../artifacts/bi_exports/compare_all_rules_vs_shap.csv

=== PREVIEW ===


,campaign_id,first_touch,last_touch,linear,time_decay_d0.5,model_credit_pos,model_credit_raw,share_pos,share_raw,shapley_credit,...,roi__linear,roi__time_decay_d0.5,roi__spend_x,roi__impressions,roi__clicks,roi__spend_y,roi__conv_last_touch,roi__model_credit_pos,roi__model_credit_raw,roi__shapley
0,10002432,383.0,313.0,348.152874,327.938405,0.0,0.0,0.0,0.0,1.718179,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,10013589,636.0,779.0,687.985070,721.936078,0.0,0.0,0.0,0.0,3.411705,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,10055720,51.0,42.0,46.343687,44.919401,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,10123946,110.0,103.0,107.296516,105.428308,0.0,0.0,0.0,0.0,0.677354,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,10245630,316.0,353.0,328.213930,339.249509,0.0,0.0,0.0,0.0,3.074691,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


,channel,first_touch,last_touch,linear,time_decay_d0.5,spend_x,impressions,clicks,spend_y,conv_last_touch,...,roi__linear,roi__time_decay_d0.5,roi__spend_x,roi__impressions,roi__clicks,roi__spend_y,roi__conv_last_touch,roi__model_credit_pos,roi__model_credit_raw,roi__shapley
0,(unassigned),438730.0,438730.0,438730.0,438730.0,4829.34054,16468027.0,5947563.0,4829.34054,806196.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


,roi__first_touch,roi__last_touch,roi__linear,roi__time_decay_d0.5,roi__spend_x,roi__impressions,roi__clicks,roi__spend_y,roi__conv_last_touch,roi__model_credit_pos,roi__model_credit_raw,roi__shapley
channel,,,,,,,,,,,,
(unassigned),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0



Top gainers vs last-touch:


,campaign_id,share_pos,last_touch,delta_last_touch_vs_shap_pos
501,32368244,0.078801,0.043131,0.035670
7,10341182,0.080588,0.049864,0.030723
122,15184511,0.075601,0.045326,0.030275
664,9100693,0.072634,0.043302,0.029331
147,16184517,0.039572,0.016926,0.022646
400,2869134,0.046205,0.024343,0.021862
10,10474106,0.020469,0.007237,0.013232
323,25920690,0.026198,0.013375,0.012823
663,9100692,0.028515,0.016682,0.011832
366,27118781,0.012653,0.002981,0.009672



Top losers vs last-touch:


,campaign_id,share_pos,last_touch,delta_last_touch_vs_shap_pos
552,5061834,2.322176e-06,0.014743,-0.014740
202,18975823,2.598448e-04,0.011415,-0.011155
179,17686799,3.896744e-06,0.008493,-0.008489
422,29427842,7.636838e-03,0.016092,-0.008455
459,30491418,3.279320e-07,0.008037,-0.008037
585,6686701,1.530402e-04,0.006241,-0.006088
156,16823030,4.748079e-04,0.006104,-0.005629
105,14121532,8.078453e-03,0.012839,-0.004761
596,7061828,1.472125e-03,0.005901,-0.004429
453,30368132,9.875719e-05,0.004454,-0.004355


In [2]:
# Build a draft campaign→channel mapping with helper stats
import os, pandas as pd, numpy as np

path_j = "../data/processed/journeys_all.parquet"
path_map = "../data/meta/campaign_channels.csv"
os.makedirs("../data/meta", exist_ok=True)

J = pd.read_parquet(path_j)

def to_pylist(x):
    if isinstance(x, list): return x
    if isinstance(x, (np.ndarray, pd.Series)): return x.tolist()
    if isinstance(x, str) and x.strip().startswith('[') and x.strip().endswith(']'):
        import ast
        try: return ast.literal_eval(x)
        except Exception: return []
    if x is None or (isinstance(x, float) and pd.isna(x)) or x == "": return []
    return [x]

for c in ["campaign_seq","click_seq","cost_seq"]:
    if c in J.columns: J[c] = J[c].apply(to_pylist)
if "conversion" not in J.columns:
    if "conversion_id" in J.columns:
        J["conversion"] = (J["conversion_id"] != -1).astype(int)
    else:
        J["conversion"] = 0

def align(a, b, fill):
    b = (b or [])
    if len(b) < len(a): b = b + [fill]*(len(a)-len(b))
    elif len(b) > len(a): b = b[:len(a)]
    return a, b

rows = []
for r in J.itertuples(index=False):
    cs = getattr(r, "campaign_seq", [])
    cl = getattr(r, "click_seq", [])
    co = getattr(r, "cost_seq", [])
    cs, cl = align(cs, cl, 0)
    cs, co = align(cs, co, 0.0)
    for c_id, ck, cst in zip(cs, cl, co):
        rows.append((str(c_id), int(ck), float(cst), int(getattr(r, "conversion", 0))))

touch = pd.DataFrame(rows, columns=["campaign_id","click","cost","conv_flag"])
# impressions = number of touchpoints
imp = touch.groupby("campaign_id", as_index=False).size().rename(columns={"size":"impressions"})
clk = touch.groupby("campaign_id", as_index=False)["click"].sum().rename(columns={"click":"clicks"})
spend = touch.groupby("campaign_id", as_index=False)["cost"].sum().rename(columns={"cost":"spend"})

# simple “last-touch approximate conversions” to help you eyeball importance (optional)
# If you already exported rule-based conversions, you can join those instead.
last_touch_path = "../artifacts/bi_exports/campaign_attribution_rules.csv"
if os.path.exists(last_touch_path):
    rb = pd.read_csv(last_touch_path).rename(columns={"campaign":"campaign_id"})
    rb["campaign_id"] = rb["campaign_id"].astype(str)
    last_conv = (rb[rb["rule"].str.startswith("last_touch")]
                 .groupby("campaign_id", as_index=False)["credit"].sum()
                 .rename(columns={"credit":"conv_last_touch"}))
else:
    last_conv = pd.DataFrame(columns=["campaign_id","conv_last_touch"])

df = (imp.merge(clk, on="campaign_id", how="left")
         .merge(spend, on="campaign_id", how="left")
         .merge(last_conv, on="campaign_id", how="left"))

df["clicks"] = df["clicks"].fillna(0).astype(int)
df["spend"] = df["spend"].fillna(0.0)
df["conv_last_touch"] = df["conv_last_touch"].fillna(0.0)

# add empty channel column for you to fill
df["channel"] = "(unassigned)"

# helpful sorting for annotation
df = df.sort_values(["conv_last_touch","impressions","spend"], ascending=[False,False,False]).reset_index(drop=True)

df.to_csv(path_map, index=False)
print("Draft mapping saved ->", path_map)
df.head(10)


Draft mapping saved -> ../data/meta/campaign_channels.csv


,campaign_id,impressions,clicks,spend,conv_last_touch,channel
0,10341182,437385,194847,139.162455,21877.0,(unassigned)
1,15184511,256102,93175,153.826559,19886.0,(unassigned)
2,9100693,100926,39442,20.840788,18998.0,(unassigned)
3,32368244,161056,77584,138.096421,18923.0,(unassigned)
4,30801593,431587,168971,25.596305,11266.0,(unassigned)
5,2869134,55796,20032,3.441929,10680.0,(unassigned)
6,15398570,378464,135066,155.572414,9491.0,(unassigned)
7,5544859,50808,26242,6.640216,7510.0,(unassigned)
8,16184517,56866,14747,8.138200,7426.0,(unassigned)
9,9100692,40265,14466,6.672559,7319.0,(unassigned)


In [3]:
# Rebuild summaries using your edited campaign→channel map
import os, pandas as pd, numpy as np

BI_DIR = "../artifacts/bi_exports"
camp_path = f"{BI_DIR}/campaign_summary_all.csv"
map_path  = "../data/meta/campaign_channels.csv"

camp = pd.read_csv(camp_path, dtype={"campaign_id":str})
chmap = pd.read_csv(map_path, dtype={"campaign_id":str})

# trust your mapping for channel
camp = camp.drop(columns=["channel"], errors="ignore").merge(chmap[["campaign_id","channel"]], on="campaign_id", how="left")
camp["channel"] = camp["channel"].fillna("(unassigned)")

# identify columns
rule_cols  = [c for c in camp.columns if c not in [
    "campaign_id","channel","spend",
    "model_credit_pos","model_credit_raw",
    "shapley_credit","share_pos","share_raw","shapley_share"
]]
model_cols = [c for c in ["model_credit_pos","model_credit_raw"] if c in camp.columns]
shpy_col   = "shapley_credit" if "shapley_credit" in camp.columns else None

def safe_div(a, b):
    b = b.replace(0, np.nan)
    return (a / b).fillna(0.0)

for rc in rule_cols:
    if f"roi__{rc}" not in camp.columns:
        camp[f"roi__{rc}"] = safe_div(camp[rc], camp["spend"])
for mc in model_cols:
    if f"roi__{mc}" not in camp.columns:
        camp[f"roi__{mc}"] = safe_div(camp[mc], camp["spend"])
if shpy_col and "roi__shapley" not in camp.columns:
    camp["roi__shapley"] = safe_div(camp[shpy_col], camp["spend"])

camp.to_csv(camp_path, index=False)
print("Updated campaign summary ->", camp_path)

# Channel rollups
agg_cols = rule_cols + model_cols + ([shpy_col] if shpy_col else []) + ["spend"]
chan = camp.groupby("channel", as_index=False)[agg_cols].sum()

# ROIs by channel
for rc in rule_cols:
    chan[f"roi__{rc}"] = safe_div(chan[rc], chan["spend"])
for mc in model_cols:
    chan[f"roi__{mc}"] = safe_div(chan[mc], chan["spend"])
if shpy_col:
    chan["roi__shapley"] = safe_div(chan[shpy_col], chan["spend"])

chan_path = f"{BI_DIR}/channel_summary_all.csv"
chan.to_csv(chan_path, index=False)
print("Updated channel summary ->", chan_path)

# Heatmap
roi_cols = [c for c in chan.columns if c.startswith("roi__")]
heatmap = chan[["channel"] + roi_cols].set_index("channel")
heatmap_path = f"{BI_DIR}/channel_roi_heatmap.csv"
heatmap.to_csv(heatmap_path)
print("Updated channel ROI heatmap ->", heatmap_path)

display(chan.head(10))
display(heatmap)


Updated campaign summary -> ../artifacts/bi_exports/campaign_summary_all.csv
Updated channel summary -> ../artifacts/bi_exports/channel_summary_all.csv
Updated channel ROI heatmap -> ../artifacts/bi_exports/channel_roi_heatmap.csv


,channel,first_touch,last_touch,linear,time_decay_d0.5,spend_x,impressions,clicks,spend_y,conv_last_touch,...,roi__roi__linear,roi__roi__time_decay_d0.5,roi__roi__spend_x,roi__roi__impressions,roi__roi__clicks,roi__roi__spend_y,roi__roi__conv_last_touch,roi__roi__model_credit_pos,roi__roi__model_credit_raw,roi__roi__shapley
0,(unassigned),438730.0,438730.0,438730.0,438730.0,4829.34054,16468027.0,5947563.0,4829.34054,806196.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


,roi__first_touch,roi__last_touch,roi__linear,roi__time_decay_d0.5,roi__spend_x,roi__impressions,roi__clicks,roi__spend_y,roi__conv_last_touch,roi__model_credit_pos,...,roi__roi__linear,roi__roi__time_decay_d0.5,roi__roi__spend_x,roi__roi__impressions,roi__roi__clicks,roi__roi__spend_y,roi__roi__conv_last_touch,roi__roi__model_credit_pos,roi__roi__model_credit_raw,roi__roi__shapley
channel,,,,,,,,,,,,,,,,,,,,,
(unassigned),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [4]:
# (Campaign-level only) + Rules vs SHAP Comparison 
import os, numpy as np, pandas as pd

BI_DIR = "../artifacts/bi_exports"
os.makedirs(BI_DIR, exist_ok=True)

PATH_RULES    = f"{BI_DIR}/campaign_attribution_rules.csv"      # from rule-based notebook
PATH_MODEL    = f"{BI_DIR}/model_based_attribution_shap.csv"    # from model-based (SHAP) notebook
PATH_JOURNEYS = "../data/processed/journeys_all.parquet"        # all journeys (0/1)

# ----------------- Load attribution outputs -----------------
rules = pd.read_csv(PATH_RULES).rename(columns={"campaign":"campaign_id"})  # [rule, campaign_id, credit]
model = pd.read_csv(PATH_MODEL)                                             # [campaign_id, credit_pos_total, credit_raw_total, share_pos, share_raw]

for df in (rules, model):
    df["campaign_id"] = df["campaign_id"].astype(str)

# Rules → wide (each rule col = attributed conversions under that rule)
rules_wide = (rules.groupby(["rule","campaign_id"], as_index=False)["credit"].sum()
                    .pivot(index="campaign_id", columns="rule", values="credit")
                    .fillna(0.0)
                    .reset_index())
rules_wide.columns.name = None


preferred_rules = ["first_touch","last_touch","linear","time_decay_d0.5"]
existing_rules  = [c for c in preferred_rules if c in rules_wide.columns]
other_rules     = [c for c in rules_wide.columns if c not in ["campaign_id"] + existing_rules]
rule_cols_final = existing_rules + other_rules  # this preserves the four if available

# Model view (both positive & raw totals + shares)
model_view = (model[["campaign_id","credit_pos_total","credit_raw_total","share_pos","share_raw"]]
              .rename(columns={"credit_pos_total":"model_credit_pos",
                               "credit_raw_total":"model_credit_raw"}))

#  Spend per campaign from journeys
J = pd.read_parquet(PATH_JOURNEYS)

def to_pylist(x):
    if isinstance(x, list): return x
    if isinstance(x, (np.ndarray, pd.Series)): return x.tolist()
    if isinstance(x, str) and x.strip().startswith('[') and x.strip().endswith(']'):
        import ast
        try: return ast.literal_eval(x)
        except Exception: return []
    if x is None or (isinstance(x, float) and pd.isna(x)) or x == "": return []
    return [x]

for c in ["campaign_seq","cost_seq"]:
    if c in J.columns: J[c] = J[c].apply(to_pylist)

def align(a, b, fill=0.0):
    if len(b) < len(a): b = b + [fill]*(len(a)-len(b))
    elif len(b) > len(a): b = b[:len(a)]
    return a, b

rows = []
for r in J.itertuples(index=False):
    cs = getattr(r, "campaign_seq", [])
    co = getattr(r, "cost_seq", [])
    cs, co = align(cs, co, 0.0)
    for c_id, cst in zip(cs, co):
        rows.append((str(c_id), float(cst)))

spend = (pd.DataFrame(rows, columns=["campaign_id","cost"])
         .groupby("campaign_id", as_index=False)["cost"].sum()
         .rename(columns={"cost":"spend"}))

# Campaign-level master (no channels) + ROI 
# union of campaigns everywhere
all_ids = pd.Series(pd.concat([
    rules_wide["campaign_id"], model_view["campaign_id"], spend["campaign_id"]
]).unique(), name="campaign_id").to_frame()

camp = (all_ids
        .merge(rules_wide[["campaign_id"] + rule_cols_final], on="campaign_id", how="left")
        .merge(model_view, on="campaign_id", how="left")
        .merge(spend, on="campaign_id", how="left"))

# fill numeric NaNs
num_cols = [c for c in camp.columns if c != "campaign_id"]
camp[num_cols] = camp[num_cols].fillna(0.0)

def safe_div(a, b):
    b = b.replace(0, np.nan)
    return (a / b).fillna(0.0)

# ROI = conversions / spend (proxy if you don't have revenue)
for rc in rule_cols_final:
    camp[f"roi__{rc}"] = safe_div(camp[rc], camp["spend"])

for mc in ["model_credit_pos","model_credit_raw"]:
    if mc in camp.columns:
        camp[f"roi__{mc}"] = safe_div(camp[mc], camp["spend"])

camp_out = f"{BI_DIR}/campaign_summary_campaign_only.csv"
camp.to_csv(camp_out, index=False)
print("Saved ->", camp_out)

# Rules vs SHAP shares + deltas 
# Build rule shares (normalize each rule to 1.0)
rules_shares = rules.copy()
rules_shares["rule_share"] = rules_shares.groupby("rule")["credit"].transform(lambda x: x / x.sum())

# campaigns × rules (shares, wide)
rule_share_wide = (rules_shares
                   .pivot(index="campaign_id", columns="rule", values="rule_share")
                   .fillna(0.0)
                   .reset_index())

# Ensure classic columns are present (fill if missing)
for col in preferred_rules:
    if col not in rule_share_wide.columns:
        rule_share_wide[col] = 0.0

# Join SHAP shares from model csv
cmp = (rule_share_wide[["campaign_id"] + preferred_rules]
       .merge(model_view[["campaign_id","share_pos","share_raw"]], on="campaign_id", how="left")
       .fillna(0.0))

# Deltas vs SHAP-positive
cmp["delta_last_touch_vs_shap_pos"]  = cmp["share_pos"] - cmp["last_touch"]
cmp["delta_first_touch_vs_shap_pos"] = cmp["share_pos"] - cmp["first_touch"]
cmp["delta_linear_vs_shap_pos"]      = cmp["share_pos"] - cmp["linear"]
if "time_decay_d0.5" in cmp.columns:
    cmp["delta_time_decay_d0.5_vs_shap_pos"] = cmp["share_pos"] - cmp["time_decay_d0.5"]

cmp_out = f"{BI_DIR}/compare_all_rules_vs_shap.csv"
cmp.to_csv(cmp_out, index=False)
print("Saved ->", cmp_out)

# Quick previews 
print("\n=== PREVIEW: campaign summary (top 5) ===")
display(camp[["campaign_id"] + rule_cols_final + ["model_credit_pos","model_credit_raw","spend"] +
             [c for c in camp.columns if c.startswith("roi__")]].head(5))

print("\n=== PREVIEW: rules vs SHAP shares + deltas (top gainers vs last-touch) ===")
if "delta_last_touch_vs_shap_pos" in cmp.columns:
    top_gain = cmp.sort_values("delta_last_touch_vs_shap_pos", ascending=False).head(10)
    top_loss = cmp.sort_values("delta_last_touch_vs_shap_pos", ascending=True).head(10)
    display(top_gain[["campaign_id","share_pos","last_touch","delta_last_touch_vs_shap_pos"]])
    display(top_loss[["campaign_id","share_pos","last_touch","delta_last_touch_vs_shap_pos"]])
else:
    display(cmp.head(10))


Saved -> ../artifacts/bi_exports/campaign_summary_campaign_only.csv
Saved -> ../artifacts/bi_exports/compare_all_rules_vs_shap.csv

=== PREVIEW: campaign summary (top 5) ===


,campaign_id,first_touch,last_touch,linear,time_decay_d0.5,model_credit_pos,model_credit_raw,spend,roi__first_touch,roi__last_touch,roi__linear,roi__time_decay_d0.5,roi__model_credit_pos,roi__model_credit_raw
0,10002432,383.0,313.0,348.152874,327.938405,0.0,0.0,6.112713,62.656299,51.204756,56.955536,53.648581,0.0,0.0
1,10013589,636.0,779.0,687.985070,721.936078,0.0,0.0,1.885498,337.311412,413.153443,364.882414,382.888801,0.0,0.0
2,10055720,51.0,42.0,46.343687,44.919401,0.0,0.0,1.142393,44.643145,36.764943,40.567214,39.320457,0.0,0.0
3,10123946,110.0,103.0,107.296516,105.428308,0.0,0.0,1.100771,99.929922,93.570745,97.473932,95.776751,0.0,0.0
4,10245630,316.0,353.0,328.213930,339.249509,0.0,0.0,2.612494,120.957235,135.119949,125.632435,129.856590,0.0,0.0



=== PREVIEW: rules vs SHAP shares + deltas (top gainers vs last-touch) ===


,campaign_id,share_pos,last_touch,delta_last_touch_vs_shap_pos
501,32368244,0.078801,0.043131,0.035670
7,10341182,0.080588,0.049864,0.030723
122,15184511,0.075601,0.045326,0.030275
664,9100693,0.072634,0.043302,0.029331
147,16184517,0.039572,0.016926,0.022646
400,2869134,0.046205,0.024343,0.021862
10,10474106,0.020469,0.007237,0.013232
323,25920690,0.026198,0.013375,0.012823
663,9100692,0.028515,0.016682,0.011832
366,27118781,0.012653,0.002981,0.009672


,campaign_id,share_pos,last_touch,delta_last_touch_vs_shap_pos
552,5061834,2.322176e-06,0.014743,-0.014740
202,18975823,2.598448e-04,0.011415,-0.011155
179,17686799,3.896744e-06,0.008493,-0.008489
422,29427842,7.636838e-03,0.016092,-0.008455
459,30491418,3.279320e-07,0.008037,-0.008037
585,6686701,1.530402e-04,0.006241,-0.006088
156,16823030,4.748079e-04,0.006104,-0.005629
105,14121532,8.078453e-03,0.012839,-0.004761
596,7061828,1.472125e-03,0.005901,-0.004429
453,30368132,9.875719e-05,0.004454,-0.004355
